# Data Exploration: CIMA Drug Interaction Dataset

This notebook explores the pharmaceutical dataset extracted from the CIMA (Centro de Información de Medicamentos) database.

**Objective:** Understand the structure, volume, and characteristics of the data loaded into MongoDB and Neo4j.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from pymongo import MongoClient
from dotenv import load_dotenv
import os

load_dotenv('../.env')

client = MongoClient(os.getenv('mongodb_uri'))
db = client[os.getenv('mongodb_db', 'drug_interaction_analysis_2')]

print('Connected to MongoDB')
print(f'Database: {db.name}')

## 1. Dataset Overview

Summary of all collections and their document counts.

In [ ]:
collections = [
    'drugs', 'drug_interactions', 'active_ingredients', 'laboratories',
    'atc_codes', 'pharmaceutical_forms', 'administration_routes',
    'excipients', 'package_types', 'content_units', 'registration_statuses',
    'dcsa', 'dcp', 'dcpf'
]

overview = []
for col in collections:
    count = db[col].count_documents({})
    overview.append({'Collection': col, 'Documents': count})

df_overview = pd.DataFrame(overview)
df_overview['Documents'] = df_overview['Documents'].apply(lambda x: f'{x:,}')
print('=== DATASET OVERVIEW ===')
print(df_overview.to_string(index=False))
print(f'\nTotal collections: {len(collections)}')

## 2. Drug Data Analysis

Exploring the structure and characteristics of the drugs collection.

In [ ]:
# Sample drug document structure
sample_drug = db['drugs'].find_one()
print('=== SAMPLE DRUG FIELDS ===')
for key in sample_drug.keys():
    val = sample_drug[key]
    val_type = type(val).__name__
    if isinstance(val, list):
        print(f'  {key}: list ({len(val)} items)')
    elif isinstance(val, dict):
        print(f'  {key}: dict ({len(val)} keys)')
    else:
        print(f'  {key}: {val_type}')

In [ ]:
# Drug classification statistics
pipeline = [
    {'$group': {
        '_id': None,
        'total': {'$sum': 1},
        'comercializado': {'$sum': {'$cond': ['$comercializado', 1, 0]}},
        'generico': {'$sum': {'$cond': ['$clasificacion.generico', 1, 0]}},
        'requiere_receta': {'$sum': {'$cond': ['$clasificacion.requiere_receta', 1, 0]}},
        'uso_hospitalario': {'$sum': {'$cond': ['$clasificacion.uso_hospitalario', 1, 0]}},
        'psicotropo': {'$sum': {'$cond': ['$clasificacion.psicotropo', 1, 0]}},
        'estupefaciente': {'$sum': {'$cond': ['$clasificacion.estupefaciente', 1, 0]}},
        'biosimilar': {'$sum': {'$cond': ['$clasificacion.biosimilar', 1, 0]}},
        'afecta_conduccion': {'$sum': {'$cond': ['$clasificacion.afecta_conduccion', 1, 0]}},
    }}
]

result = list(db['drugs'].aggregate(pipeline))[0]
total = result['total']

print('=== DRUG CLASSIFICATION ===')
for key in ['comercializado', 'generico', 'requiere_receta', 'uso_hospitalario',
            'psicotropo', 'estupefaciente', 'biosimilar', 'afecta_conduccion']:
    count = result[key]
    pct = count / total * 100
    print(f'  {key:25s}: {count:6,} ({pct:5.1f}%)')

In [ ]:
# Top 15 laboratories by number of drugs
pipeline = [
    {'$group': {'_id': '$laboratorio_titular.nombre', 'count': {'$sum': 1}}},
    {'$sort': {'count': -1}},
    {'$limit': 15}
]

top_labs = list(db['drugs'].aggregate(pipeline))
df_labs = pd.DataFrame(top_labs).rename(columns={'_id': 'Laboratory', 'count': 'Drugs'})
print('=== TOP 15 LABORATORIES BY DRUG COUNT ===')
print(df_labs.to_string(index=False))

In [ ]:
# Top 15 ATC codes
pipeline = [
    {'$group': {'_id': '$atc.codigo', 'count': {'$sum': 1}}},
    {'$sort': {'count': -1}},
    {'$limit': 15}
]

top_atc = list(db['drugs'].aggregate(pipeline))
df_atc = pd.DataFrame(top_atc).rename(columns={'_id': 'ATC Code', 'count': 'Drugs'})
print('=== TOP 15 ATC CODES ===')
print(df_atc.to_string(index=False))

In [ ]:
# Top 15 active ingredients
pipeline = [
    {'$unwind': '$formas_farmaceuticas'},
    {'$unwind': '$formas_farmaceuticas.composicion'},
    {'$group': {
        '_id': '$formas_farmaceuticas.composicion.principio_activo.nombre',
        'count': {'$sum': 1}
    }},
    {'$sort': {'count': -1}},
    {'$limit': 15}
]

top_ingredients = list(db['drugs'].aggregate(pipeline))
df_ing = pd.DataFrame(top_ingredients).rename(columns={'_id': 'Active Ingredient', 'count': 'Appearances'})
print('=== TOP 15 ACTIVE INGREDIENTS ===')
print(df_ing.to_string(index=False))

## 3. Drug Interactions Analysis

Exploring the interactions dataset — the core of this TFM.

In [ ]:
# Interaction overview
total_interactions = db['drug_interactions'].count_documents({})
print(f'Total drug interactions: {total_interactions:,}')

# Sample interaction
sample = db['drug_interactions'].find_one()
print(f'\n=== SAMPLE INTERACTION ===')
print(f'Source: {sample["medicamento_origen"]["nombre"]}')
print(f'Target: {sample["medicamento_destino"]["nombre"]}')
print(f'Effect: {sample["interaccion"]["efecto"]}')
print(f'Recommendation: {sample["interaccion"]["recomendacion"]}')

In [ ]:
# Drugs with most interactions
pipeline = [
    {'$group': {'_id': '$medicamento_origen.nombre', 'interactions': {'$sum': 1}}},
    {'$sort': {'interactions': -1}},
    {'$limit': 15}
]

top_drugs = list(db['drug_interactions'].aggregate(pipeline))
df_top = pd.DataFrame(top_drugs).rename(columns={'_id': 'Drug', 'interactions': 'Interactions'})
print('=== TOP 15 DRUGS BY INTERACTION COUNT ===')
print(df_top.to_string(index=False))

In [ ]:
# Interaction text length analysis
pipeline = [
    {'$project': {
        'efecto_len': {'$strLenCP': {'$ifNull': ['$interaccion.efecto', '']}},
        'recomendacion_len': {'$strLenCP': {'$ifNull': ['$interaccion.recomendacion', '']}}
    }},
    {'$group': {
        '_id': None,
        'avg_efecto': {'$avg': '$efecto_len'},
        'max_efecto': {'$max': '$efecto_len'},
        'avg_recomendacion': {'$avg': '$recomendacion_len'},
        'max_recomendacion': {'$max': '$recomendacion_len'},
    }}
]

text_stats = list(db['drug_interactions'].aggregate(pipeline))[0]
print('=== INTERACTION TEXT STATISTICS ===')
print(f'Effect text:         avg {text_stats["avg_efecto"]:.0f} chars, max {text_stats["max_efecto"]} chars')
print(f'Recommendation text: avg {text_stats["avg_recomendacion"]:.0f} chars, max {text_stats["max_recomendacion"]} chars')

## 4. Summary

The CIMA dataset provides a comprehensive source of Spanish pharmaceutical data with:
- ~30,000 medications from authorized Spanish pharmacies
- ~70,000 drug-drug interactions with textual descriptions in Spanish
- Rich metadata: ATC codes, laboratories, active ingredients, pharmaceutical forms
- Text fields suitable for NLP extraction (severity, interaction type, mechanism)

In [ ]:
client.close()
print('MongoDB connection closed.')